# Scientific metadata with Ontolutils

To follow this tutorial before the 1.0 release, run `python -m pip install .` from the repository root and select that Python environment as your Jupyter kernel. Run the cells in order. The first section writes `measurement.jsonld` in the notebook's working directory. The final section requires internet access.

In [ ]:
from typing import Optional
from pydantic import Field, ValidationError
from ontolutils import Thing, QUDT_UNIT, namespaces, urirefs
from ontolutils.typing import AnyIri

@namespaces(ex="https://example.org/vocab/")
@urirefs(Measurement="ex:Measurement", value="ex:value", unit="ex:unit", note="ex:note")
class Measurement(Thing):
    value: float = Field(ge=0, alias="measured_value")
    unit: AnyIri
    note: Optional[str] = None

measurement = Measurement(
    id="https://example.org/measurements/1",
    measured_value=1.5,
    unit=str(QUDT_UNIT.M_PER_SEC),
    label="Velocity magnitude",
)
assert measurement.value == 1.5

`value` and `unit` are required. `note` can be omitted or set to `None`. The alias `measured_value` is an alternative Python input name; the RDF predicate remains `ex:value`. The example vocabulary should be replaced by your community's terms.

In [ ]:
try:
    measurement.value = -1
except ValidationError as error:
    print(error.errors()[0]["loc"])
else:
    raise AssertionError("Expected a validation error")
assert measurement.value == 1.5

## Save and reload

JSON-LD is JSON with RDF context. `model_dump_jsonld()` returns text; `get_jsonld_dict()` returns a dictionary. A loader normally returns a list; use `limit=1` for a single match.

In [ ]:
from pathlib import Path

document = measurement.model_dump_jsonld()
Path("measurement.jsonld").write_text(document, encoding="utf-8")
restored = Measurement.from_jsonld(source="measurement.jsonld", limit=1)
assert restored.value == 1.5
assert str(restored.unit) == str(QUDT_UNIT.M_PER_SEC)
print(measurement.serialize("ttl"))

## Inspect the RDF graph

Compare triples rather than serializer whitespace or prefix order. Keep stable IDs when sharing metadata across files.

In [ ]:
from rdflib import Graph, URIRef
from rdflib.compare import isomorphic

original_graph = Graph().parse(data=document, format="json-ld")
restored_graph = Graph().parse(data=restored.serialize("ttl"), format="ttl")
assert isomorphic(original_graph, restored_graph)
assert float(original_graph.value(URIRef(measurement.id), URIRef(Measurement.iri("value")))) == 1.5

## Load models from a published ontology

The project-defined model above lets you choose fields and validation rules explicitly. Here, Metadata4Ing (M4I) supplies class and property declarations from which Ontolutils generates models. Inspect `Tool.rdf_fields()` to see the Python field names and their RDF predicates. M4I declares that `hasParameter` applies to tools and methods.

In [ ]:
from ontolutils import onto
from rdflib import RDF

m4i = onto.load("https://w3id.org/nfdi4ing/metadata4ing/ontology.ttl")
Tool = m4i.models.Tool
tool_id = "https://example.org/instruments/laser"
parameter_id = "https://example.org/parameters/power"
tool = Tool(id=tool_id, hasParameter=parameter_id)

print(Tool.rdf_fields())
turtle = tool.serialize("ttl")
print(turtle)

graph = Graph().parse(data=turtle, format="ttl")
assert (URIRef(tool_id), RDF.type, URIRef(Tool.iri())) in graph
assert (URIRef(tool_id), URIRef(Tool.rdf_fields()["hasParameter"]), URIRef(parameter_id)) in graph

Generated fields reflect the declarations in the loaded document. Use a profile or your own model for application-specific Python validation, and SHACL for RDF constraints. For reproducible or offline use, keep a verified copy of the ontology and load its local path. See the [ontology loader guide](../docs/userguide/ontology_loader.rst).